## Lab 5: AgentCore Evaluations - Online Evaluation for Customer Support Agent

### Overview

This lab demonstrates how to use AgentCore Evaluations to continuously monitor your production customer support agent from Lab 4. You'll configure online evaluation to automatically assess agent performance in real-time as customers interact with it.

**Workshop Journey:**

- **Lab 1 (Done):** Create Agent Prototype - Built a functional customer support agent
- **Lab 2 (Done):** Enhance with Memory - Added conversation context and personalization
- **Lab 3 (Done):** Scale with Gateway & Identity - Shared tools across agents securely
- **Lab 4 (Done):** Deploy to Production - Used AgentCore Runtime with observability
- **Lab 5 (Current):** Evaluate Agent Performance - Monitor quality with online evaluations
- **Lab 6:** Build User Interface - Create a customer-facing application

### What You'll Learn

You'll configure online evaluation with built-in evaluators, generate test interactions, and analyze quality metrics through AgentCore Observability dashboards to improve agent performance.

### Online Evaluation Overview

Online evaluation continuously monitors deployed agents in production, unlike on-demand evaluation which analyzes specific selected interactions. It consists of three components: session sampling with configurable rules, multiple evaluation methods (built-in or custom evaluators), and monitoring through dashboards with quality trends and low-scoring session investigation.

Since your agent runs on AgentCore Runtime, AgentCore Observability automatically instruments the code and provides comprehensive logs and traces using [OTEL](https://opentelemetry.io/) instrumentation.

### Prerequisites

Complete Lab 4 to have the customer support agent deployed. You'll need AWS account access to Amazon Bedrock AgentCore with Evaluations permissions.

### Architecture
<div style="text-align:left">
    <img src="images/architecture_lab5_evaluation.png" width="75%"/>
</div>

*Online evaluation automatically monitors agent interactions, applies evaluators based on sampling rules, and outputs results to CloudWatch for analysis.*

### Step 1: Import Required Libraries and Initialize Clients

In [1]:
from bedrock_agentcore_starter_toolkit import Evaluation, Runtime
import json
import uuid
from pathlib import Path
from boto3.session import Session
from IPython.display import Markdown, display
from lab_helpers.utils import get_ssm_parameter, get_or_create_cognito_pool

In [2]:
boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

Region: us-west-2


In [3]:
eval_client = Evaluation(region=region)
runtime_client = Runtime()

### Step 2: Retrieve Agent Information from Lab 4

Retrieve the customer support agent ARN from SSM Parameter Store where it was saved during Lab 4 deployment.

In [4]:
try:
    # Get agent ARN from SSM parameter store (saved in Lab 4)
    agent_arn = get_ssm_parameter("/app/customersupport/agentcore/runtime_arn")
    
    # Extract agent ID from ARN
    agent_id = agent_arn.split(":")[-1].split("/")[-1]
    
    # Set runtime client config path
    runtime_client._config_path = Path.cwd() / ".bedrock_agentcore.yaml"
    
    print("Agent ID:", agent_id)
    print("Agent ARN:", agent_arn)
except Exception as e:
    raise Exception(f"""Missing agent information from Lab 4. Please run lab-04-agentcore-runtime.ipynb first. Error: {str(e)}""")

Agent ID: customer_support_agent-1nAFU5HPLi
Agent ARN: arn:aws:bedrock-agentcore:us-west-2:818158348946:runtime/customer_support_agent-1nAFU5HPLi


### Step 3: Create Online Evaluation Configuration

Now let's create an online evaluation configuration for our customer support agent. We'll use built-in evaluators to assess different aspects of agent performance:

- **Builtin.GoalSuccessRate** - Measures how well the agent achieves user goals
- **Builtin.Correctness** - Evaluates factual accuracy of responses
- **Builtin.ToolSelectionAccuracy** - Evaluates appropriate tool selection

We'll set the sampling rate to 100% for demonstration purposes, but in production you might use a lower rate (e.g., 10-20%) based on your traffic volume.

In [5]:
response = eval_client.create_online_config(
    agent_id=agent_id,
    config_name="customer_support_agent_eval",
    sampling_rate=100,  # Evaluate 100% of sessions for demo
    evaluator_list=[
        "Builtin.GoalSuccessRate", 
        "Builtin.Correctness",
        "Builtin.ToolSelectionAccuracy"
    ],
    config_description="Customer support agent online evaluation",
    auto_create_execution_role=True
)

print("Online evaluation configuration created successfully!")
print(f"Configuration ID: {response['onlineEvaluationConfigId']}")

Creating online evaluation config: customer_support_agent_eval for agent: customer_support_agent-1nAFU5HPLi
Configuration: sampling_rate=100.0%, evaluators=['Builtin.GoalSuccessRate', 'Builtin.Correctness', 'Builtin.ToolSelectionAccuracy']
Creating online evaluation config: customer_support_agent_eval for agent: customer_support_agent-1nAFU5HPLi
Auto-creating execution role for config: customer_support_agent_eval
Getting or creating evaluation execution role for config: customer_support_agent_eval
Using AWS region: us-west-2, account ID: 818158348946
Role name: AgentCoreEvalsSDK-us-west-2-d04ba7b68b
Role doesn't exist, creating new evaluation execution role: AgentCoreEvalsSDK-us-west-2-d04ba7b68b
Creating IAM role: AgentCoreEvalsSDK-us-west-2-d04ba7b68b
✓ Role created: arn:aws:iam::818158348946:role/AgentCoreEvalsSDK-us-west-2-d04ba7b68b
✓ Execution policy attached: AgentCoreEvaluationPolicy-us-west-2-d04ba7b68b
Waiting for IAM role propagation...
Role creation complete and ready for u

✅ Online evaluation configuration created!

Online evaluation configuration created successfully!
Configuration ID: customer_support_agent_eval-ShKEIBFink


### Step 4: Verify Configuration Status

Verify the evaluation configuration is properly created and enabled by retrieving its details.

In [6]:
config_details = eval_client.get_online_config(config_id=response['onlineEvaluationConfigId'])
print("Configuration Details:")
print(json.dumps(config_details, indent=2, default=str))

Configuration Details:
{
  "ResponseMetadata": {
    "RequestId": "5d5593ef-b7cb-4089-aa11-f32fad00a1a7",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 28 Apr 2026 18:56:42 GMT",
      "content-type": "application/json",
      "content-length": "1050",
      "connection": "keep-alive",
      "x-amzn-requestid": "5d5593ef-b7cb-4089-aa11-f32fad00a1a7",
      "x-amz-apigw-id": "ci1kwG4TPHcETUA=",
      "x-amzn-trace-id": "Root=1-69f102ea-0e4203591e20fc9e2a6bbafd"
    },
    "RetryAttempts": 0
  },
  "onlineEvaluationConfigArn": "arn:aws:bedrock-agentcore:us-west-2:818158348946:online-evaluation-config/customer_support_agent_eval-ShKEIBFink",
  "onlineEvaluationConfigId": "customer_support_agent_eval-ShKEIBFink",
  "onlineEvaluationConfigName": "customer_support_agent_eval",
  "description": "Customer support agent online evaluation",
  "rule": {
    "samplingConfig": {
      "samplingPercentage": 100.0
    }
  },
  "dataSourceConfig": {
    "cloudWatchLogs": {
      

### Step 5: Generate Test Interactions

Invoke the customer support agent with various queries to generate traces for evaluation. Different test scenarios will demonstrate how the evaluators assess agent performance.

In [7]:
# Get authentication token
access_token = get_or_create_cognito_pool(refresh_token=True)
print(f"Access token obtained: {access_token['bearer_token'][:20]}...")

def invoke_agent_runtime(prompt, session_id=None):
    """Invoke the agent runtime using starter toolkit"""
    if not session_id:
        session_id = str(uuid.uuid4())
    
    response = runtime_client.invoke(
        payload={"prompt": prompt},
        session_id=session_id,
        bearer_token=access_token['bearer_token']
    )
    
    return response, session_id

Access token obtained: eyJraWQiOiJIU0hXZUpj...


#### Test Scenario 1: Product Information Query

In [8]:
session1 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I need information about the Gaming Console Pro. What are its specifications and price?",
    session1
)
print("Customer Query: Product information request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Product information request


"I understand you're looking for information about the Gaming Console Pro, including its specifications and price. Unfortunately, I'm unable to retrieve the technical specifications at this moment as the system indicates they're currently not available.

However, I can help you in a few ways:

1. **Contact our Technical Support Team**: They have access to the most current and detailed product information, including full specifications, compatibility details, and current pricing.

2. **Check Current Pricing**: While I can't provide the exact price right now, I can help you look up current pricing and deals if you'd like.

3. **Alternative Options**: If you're interested in similar gaming consoles, I can provide information about other gaming console models we carry.

Would you like me to help you connect with our technical support team for detailed specifications, or would you prefer to look up current pricing and deals for the Gaming Console Pro? I can also help you explore alternative gaming console options if you're interested."

#### Test Scenario 2: Technical Support Request

In [9]:
session2 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "My laptop won't start up. Can you help me troubleshoot this issue?",
    session2
)
print("Customer Query: Technical support request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Technical support request


" 

I can definitely help you troubleshoot your laptop startup issue. To provide the best assistance, I'll need some additional information:

1. What is the make and model of your laptop? (For example: Dell XPS15, MacBook Pro 16-inch, Lenovo ThinkPad X1 Carbon, etc.)

2. When you try to start it, do you see any specific error messages, lights, or behaviors? For example:
   - Does it power on but not display anything?
   - Do you see any error codes or flashing lights?
   - Does it start to boot but then stop or freeze?

3. Have you made any recent changes to the laptop before this started happening? (For example: software updates, hardware connections/disconnections, etc.)

4. How old is the laptop, and what's the operating system (Windows, macOS, Linux)?

With these details, I can provide specific troubleshooting steps tailored to your situation. Would you like to share this information so I can help you get your laptop running again?"

#### Test Scenario 3: Return Policy Inquiry

In [10]:
session3 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I bought a smartphone last week but it's not working properly. What's your return policy?",
    session3
)
print("Customer Query: Return policy inquiry")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Return policy inquiry


"Perfect! Here are our smartphone return policy details:

**Smartphone Return Policy:**
- **Return Window:** 30 days from delivery (since you purchased it last week, you're well within this period)
- **Condition Requirements:** Must be in original packaging, no physical damage, and factory reset required
- **Return Process:** You can use our online RMA portal or contact technical support
- **Refund Timeline:** 5-7 business days after inspection
- **Shipping:** We provide free return shipping with a prepaid label
- **Warranty:** Includes a 1-year manufacturer warranty

Since your smartphone is under 30 days old, you're eligible for a full return. Would you like me to help you initiate the return process? I can either:
1. Guide you through our online RMA portal, or 
2. Connect you with our technical support team who can help process the return and send you a prepaid return label.

Which option would you prefer?"

#### Test Scenario 4: Complex Multi-Tool Query

In [11]:
session4 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I need help with my Gaming Console Pro. First, can you tell me about its warranty? Then I need technical support for connection issues.",
    session4
)
print("Customer Query: Complex multi-tool request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Complex multi-tool request


"# Gaming Console Pro Support

## Warranty Information
Based on the serial number **MNO33333333**, your Gaming Console Pro warranty has **expired**. Here are the details:

- **Purchase Date**: November 25, 2023
- **Warranty End Date**: November 25, 2024
- **Status**: Expired (520 days ago)
- **Coverage Type**: Gaming Warranty
- **Covered Issues**: Controller issues, overheating protection, and hard drive replacement

Since your warranty has expired, you can still get repair service, but it will be at current service pricing rather than covered under warranty. Extended warranty options may be available - I can help you explore those if you're interested.

## Connection Issues Technical Support

For your connection issues, here are the most relevant troubleshooting steps from our technical guides:

### Wi-Fi Connection Troubleshooting
1. **Basic Checks**
   - Move closer to your router to eliminate weak signal issues
   - Verify your Wi-Fi password is entered correctly
   - Restart both your Gaming Console Pro and router

2. **Advanced Solutions**
   - Check for network interference from other devices
   - Update your router's firmware
   - Check bandwidth usage from other devices
   - Try a different network cable if using wired connection

3. **Connection Stability**
   - If connections drop, update network drivers
   - Check power saving settings on your console
   - Reset network settings as a last resort

### Bluetooth Pairing Issues (if applicable)
- Clear Bluetooth cache on both devices
- Ensure both devices are in pairing mode
- Remove and re-pair the devices
- Check battery levels on Bluetooth accessories
- Update Bluetooth drivers

### General Connectivity Tips
- Test functionality after each troubleshooting step
- Use WPA3 encryption if available for security
- Regularly update your console's firmware
- Monitor connected devices on your network

Would you like me to help you with any specific connection issue you're experiencing, or would you like information about extended warranty options?"

#### Test Scenario 5: General Capability Query

In [12]:
session5 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "What kind of support can you provide? List all your available tools and capabilities.",
    session5
)
print("Customer Query: Capability inquiry")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Capability inquiry


"  
I can help you with a variety of technical support and product-related inquiries! Here are my available tools and capabilities:

## Available Tools:

### **Product Information & Specifications**
- `get_product_info()` - Provides detailed technical specifications for electronics products including features, warranty information, and technical details

### **Warranty & Return Policies**
- `get_return_policy()` - Retrieves return policy information for specific product categories with timeframes and conditions
- `LambdaUsingSDK___check_warranty_status` - Checks warranty status using product serial number (with optional email verification)

### **Technical Support**
- `get_technical_support()` - Provides troubleshooting and technical assistance for electronic issues

### **Research & Information Retrieval**
- `LambdaUsingSDK___web_search` - Searches the web for updated technical documentation and information using DuckDuckGo

## What I Can Help You With:

1. **Product Specifications** - Get detailed technical data on laptops, smartphones, headphones, monitors, and other electronics
2. **Warranty Checks** - Verify warranty status using serial numbers
3. **Return Policies** - Understand return windows and conditions for different product categories
4. **Technical Troubleshooting** - Get support for device issues and problems
5. **Technical Documentation** - Find installation guides, user manuals, and technical resources
6. **Product Research** - Compare features and specifications across different models

## How I Can Assist You:
- Explain product features and capabilities
- Help you understand warranty coverage and return options
- Provide troubleshooting steps for technical issues
- Retrieve current technical documentation and guides
- Find answers to specific technical questions about your electronics

**What would you like help with today?** I can assist with product specifications, warranty checks, technical support, or finding documentation for your electronics."

### Step 6: Monitor Evaluation Results

Monitor evaluation results through the AgentCore Observability console. Results may take a few minutes to appear as the system processes traces and applies evaluators.

#### Accessing the Dashboard

1. Navigate to the [AgentCore Observability console](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents)
2. Find your customer support agent in the agents list
3. Click on the `DEFAULT` endpoint to view evaluation metrics
4. Look for the evaluation scores in the traces and sessions views

#### What You'll See

The dashboard will show:
- **Goal Success Rate**: How well the agent achieves customer objectives
- **Correctness**: Accuracy of information provided
- **Tool Selection Accuracy**: Appropriate tool choices for queries

![Online Evaluation Dashboard](images/online_evaluations_dashboard.png)

*Evaluation metrics displayed in the AgentCore Observability dashboard*

### Step 7: Understanding Evaluation Metrics

**Goal Success Rate** measures whether the agent successfully addresses the customer's primary intent. High scores indicate effective problem-solving; low scores suggest unmet needs, incomplete responses, or misunderstood requests.

**Correctness** evaluates factual accuracy of responses. High scores indicate accurate and reliable information; low scores suggest incorrect facts, outdated information, or misleading guidance.

**Tool Selection Accuracy** evaluates whether the agent chooses appropriate tools for each task. High scores indicate proper tool selection; low scores suggest wrong tools, unnecessary calls, or missing tool usage.

### Step 8: Analyzing Results and Next Steps

**For Low Goal Success Rates:** Refine the agent's system prompt, improve tool descriptions and parameters, and add specific training examples.

**For Low Correctness Scores:** Update the knowledge base with current information, improve fact-checking mechanisms, and review tool responses.

**For Tool-Related Issues:** Refine tool parameter schemas, improve tool selection logic, and enhance tool documentation.

**Continuous Monitoring:** Set up CloudWatch alarms for evaluation metrics, create dashboards for trend analysis, and implement automated alerts for quality degradation.

### Step 9: Clean Up (Optional)

Disable the online evaluation configuration if needed by uncommenting the code below.

In [ ]:
# Uncomment the following lines if you want to disable the evaluation configuration
# eval_client.delete_online_config(config_id=response['onlineEvaluationConfigId'])
# print("Online evaluation configuration disabled")

### Congratulations! 🎉

You have successfully completed **Lab 5: AgentCore Evaluations - Online Evaluation!**

### What You Accomplished

You configured automatic continuous online evaluation for your customer support agent with built-in evaluators assessing Goal Success Rate (customer satisfaction and problem resolution), Correctness (factual accuracy), and Tool Selection Accuracy (proper tool usage). Evaluation results are integrated with AgentCore Observability dashboards for real-time insights.

**Key Benefits:** Proactive quality assurance catches issues before customer impact, data-driven optimization guides improvements, production confidence through performance monitoring at scale, and continuous learning identifies patterns and opportunities.

**Next Steps:** Monitor your evaluation dashboard regularly, set up CloudWatch alarms for quality thresholds, use insights to iteratively improve your agent, and consider adding custom evaluators for domain-specific metrics.

### Next Up: [Lab 6: Build User Interface →](lab-06-frontend.ipynb)

Complete the customer experience by building a user-friendly web interface for customers to interact with your quality-monitored agent.

Your customer support agent is now production-ready with comprehensive quality monitoring! 🚀